In [1]:
#Перед нами стоит задача классификации.В качестве модели используем градиентный бустинг с помощью библиотеки catboost для 
#Используем исменно данную реализацию бустинга, так как она "из коробки" способна работать с категориальными перемнными
import optuna

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import precision_score,recall_score,roc_auc_score,f1_score,confusion_matrix,classification_report

In [2]:
df = pd.read_csv('clear_train.csv')
df.head()

,Unnamed: 0,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Female,49.0,Ludhiana,Working Professional,Chef,-1.0,5.0,-1.00,-1.0,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Male,26.0,Varanasi,Working Professional,Teacher,-1.0,4.0,-1.00,-1.0,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Male,33.0,Visakhapatnam,Student,Other,5.0,-1.0,8.97,2.0,-1.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Male,22.0,Mumbai,Working Professional,Teacher,-1.0,5.0,-1.00,-1.0,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Female,30.0,Kanpur,Working Professional,Business Analyst,-1.0,1.0,-1.00,-1.0,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


In [3]:
categorical_features = ['Gender',
                       'City',
                       'Working Professional or Student',
                       'Profession',
                       'Sleep Duration',
                       'Dietary Habits',
                       'Degree',
                       'Have you ever had suicidal thoughts ?',
                       'Family History of Mental Illness']


In [4]:
df.columns

Index(['Unnamed: 0', 'Gender', 'Age', 'City',
       'Working Professional or Student', 'Profession', 'Academic Pressure',
       'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
       'Sleep Duration', 'Dietary Habits', 'Degree',
       'Have you ever had suicidal thoughts ?', 'Work/Study Hours',
       'Financial Stress', 'Family History of Mental Illness', 'Depression'],
      dtype='object')

In [5]:
X = df.drop(['Unnamed: 0','Depression'],axis=1)
y = df['Depression']
X.head()

,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,Female,49.0,Ludhiana,Working Professional,Chef,-1.0,5.0,-1.00,-1.0,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No
1,Male,26.0,Varanasi,Working Professional,Teacher,-1.0,4.0,-1.00,-1.0,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No
2,Male,33.0,Visakhapatnam,Student,Other,5.0,-1.0,8.97,2.0,-1.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No
3,Male,22.0,Mumbai,Working Professional,Teacher,-1.0,5.0,-1.00,-1.0,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes
4,Female,30.0,Kanpur,Working Professional,Business Analyst,-1.0,1.0,-1.00,-1.0,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
cb_model = CatBoostClassifier()

In [16]:
cb_model.fit(X_train,y_train,cat_features=categorical_features)

Learning rate set to 0.077424
0:	learn: 0.5682297	total: 335ms	remaining: 5m 34s
1:	learn: 0.4745154	total: 493ms	remaining: 4m 5s
2:	learn: 0.4084077	total: 646ms	remaining: 3m 34s
3:	learn: 0.3551984	total: 795ms	remaining: 3m 17s
4:	learn: 0.3183929	total: 955ms	remaining: 3m 10s
5:	learn: 0.2885038	total: 1.1s	remaining: 3m 1s
6:	learn: 0.2669361	total: 1.23s	remaining: 2m 55s
7:	learn: 0.2485827	total: 1.35s	remaining: 2m 48s
8:	learn: 0.2348359	total: 1.48s	remaining: 2m 43s
9:	learn: 0.2239735	total: 1.61s	remaining: 2m 39s
10:	learn: 0.2162036	total: 1.75s	remaining: 2m 37s
11:	learn: 0.2097539	total: 1.88s	remaining: 2m 34s
12:	learn: 0.2027051	total: 2.03s	remaining: 2m 33s
13:	learn: 0.1974829	total: 2.16s	remaining: 2m 32s
14:	learn: 0.1930376	total: 2.3s	remaining: 2m 30s
15:	learn: 0.1897428	total: 2.43s	remaining: 2m 29s
16:	learn: 0.1863952	total: 2.56s	remaining: 2m 28s
17:	learn: 0.1836614	total: 2.68s	remaining: 2m 26s
18:	learn: 0.1811673	total: 2.8s	remaining: 2m 2

In [17]:
base_pred = cb_model.predict(X_test)

In [25]:
base_prec = precision_score(y_test,base_pred)
base_rec = recall_score(y_test,base_pred)
base_f1 = f1_score(y_test,base_pred)
base_roc_auc = roc_auc_score(y_test,base_pred)
conf_matrix = confusion_matrix(y_test,base_pred)
report = classification_report(y_test,base_pred)

In [28]:
print('precision =',base_prec)
print('recall =',base_rec)
print('f1 =',base_f1)
print('roc_auc =',base_roc_auc)
display(conf_matrix)
print(report)
#Видим, что базовая модель выдаёт вполне неплохие показатели всех метрик. Только в конкретном случае, хотелось бы 
#видеть recall больше для полного охвата людей с депрессией

precision = 0.8444087532623971
recall = 0.8160651920838184
f1 = 0.8299950666008881
roc_auc = 0.8911745085103682


array([[22211,   775],
       [  948,  4206]], dtype=int64)

              precision    recall  f1-score   support

           0       0.96      0.97      0.96     22986
           1       0.84      0.82      0.83      5154

    accuracy                           0.94     28140
   macro avg       0.90      0.89      0.90     28140
weighted avg       0.94      0.94      0.94     28140



In [33]:
features = cb_model.feature_names_
importance = cb_model.feature_importances_

for f,i in zip(features,importance):
    print(f,':',i)

Gender : 0.3747670689597349
Age : 29.131586595178067
City : 2.861981639920748
Working Professional or Student : 0.01642510447791969
Profession : 3.49113312226932
Academic Pressure : 6.303700409590826
Work Pressure : 8.018194365330627
CGPA : 1.2963755745924757
Study Satisfaction : 1.0432227552522815
Job Satisfaction : 7.796959367074027
Sleep Duration : 2.894196686353519
Dietary Habits : 3.9824287077569367
Degree : 2.7591506730210655
Have you ever had suicidal thoughts ? : 16.476667021098784
Work/Study Hours : 5.546103268100959
Financial Stress : 7.521986509097306
Family History of Mental Illness : 0.4851211319253858


In [34]:
#Наша модель обучалась 2 минуты 15 секунд. Понятно, что поиск гиперпараметров по сетке займёт
#очень много времени. Можно использовать либо RandomSearch, но тогда мы можем упустить лучшую вариацию гиперпараметров
#Поэтому будем использовать библиотеку optuna основанную на байесовской оптимизации
#Так же благодаря использованию optuna мы можем позволиить себе расширить диапазон гиперпараметров

In [3]:
categorical_features = [0,2,3,4,10,11,12,13,16]

In [8]:
#В силу несбалансированности классов используем балансировку
#Так как у нас много категориальных перемнных, будем варьировать параметр, отвечающий за кодировку
#Остальные варьируемы параметры классические(iterations для макс количества деревьев,learning_rate для шага градиента)
#depth для глубины деревьев и l2_leaf_reg для настройки регуляризации(борьбы с переобучением)
#Также протестируем разые метрики используемые в процессе обучения, так для нас очень важен recall в силу задачи
def objective(trial):
    params={'iterations' : trial.suggest_int('iterations',400,1600,step=100),
    'learning_rate' : trial.suggest_float('learning_rate',0.01,0.1,log=True),
    'depth' : trial.suggest_int('depth',4,10),
    'l2_leaf_reg' : trial.suggest_int('l2_leaf_reg',1,7),
    'eval_metric' : trial.suggest_categorical('eval_metric', ['AUC', 'Recall', 'Logloss'])}
    
    cbc = CatBoostClassifier(**params,auto_class_weights='Balanced',verbose=0,random_seed=42,cat_features=categorical_features)
    return cross_val_score(cbc, X_train, y_train, cv=3, scoring='recall').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=30)

best_params = study.best_params

[I 2025-11-28 15:10:25,508] A new study created in memory with name: no-name-d1101f34-f12c-4777-a02d-96858836bba4
[I 2025-11-28 15:12:26,638] Trial 0 finished with value: 0.9293590104105248 and parameters: {'iterations': 500, 'learning_rate': 0.015751454707678976, 'depth': 5, 'l2_leaf_reg': 1, 'eval_metric': 'Logloss'}. Best is trial 0 with value: 0.9293590104105248.
[I 2025-11-28 15:13:58,372] Trial 1 finished with value: 0.9288200829265406 and parameters: {'iterations': 400, 'learning_rate': 0.08324074892423197, 'depth': 4, 'l2_leaf_reg': 5, 'eval_metric': 'Recall'}. Best is trial 0 with value: 0.9293590104105248.
[I 2025-11-28 15:43:36,930] Trial 2 finished with value: 0.9296529119535079 and parameters: {'iterations': 800, 'learning_rate': 0.014309558343130706, 'depth': 6, 'l2_leaf_reg': 2, 'eval_metric': 'Recall'}. Best is trial 2 with value: 0.9296529119535079.
[I 2025-11-28 15:47:53,421] Trial 3 finished with value: 0.9291140420633984 and parameters: {'iterations': 1000, 'learnin

In [8]:
best_params = {'iterations': 1200,
 'learning_rate': 0.01830471873761281,
 'depth': 4,
 'l2_leaf_reg': 3,
 'eval_metric': 'AUC'}

In [9]:
cb_model = CatBoostClassifier(**best_params,auto_class_weights='Balanced',verbose=0,random_seed=42,cat_features=categorical_features)

In [12]:
cb_model.fit(X_train,y_train)

In [21]:
final_pred = cb_model.predict(X_test)

In [25]:
final_prec = precision_score(y_test,final_pred)
final_rec = recall_score(y_test,final_pred)
final_f1 = f1_score(y_test,final_pred)
final_roc_auc = roc_auc_score(y_test,final_pred)
conf_matrix = confusion_matrix(y_test,final_pred)
report = classification_report(y_test,final_pred)

In [27]:
print('precision =',final_prec)
print('recall =',final_rec)
print('f1 =',final_f1)
print('roc_auc =',final_roc_auc)
display(conf_matrix)
print(report)

precision = 0.7096063454759107
recall = 0.9373302289483896
f1 = 0.8077244607925096
roc_auc = 0.9256606769905091


array([[21009,  1977],
       [  323,  4831]], dtype=int64)

              precision    recall  f1-score   support

           0       0.98      0.91      0.95     22986
           1       0.71      0.94      0.81      5154

    accuracy                           0.92     28140
   macro avg       0.85      0.93      0.88     28140
weighted avg       0.93      0.92      0.92     28140



In [33]:
test_data = pd.read_csv('clear_test.csv').drop('Unnamed: 0',axis=1)
test_data
test = pd.read_csv('test.csv')

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,140700,Shivam,Male,53.0,Visakhapatnam,Working Professional,Judge,NaN,2.0,NaN,NaN,5.0,Less than 5 hours,Moderate,LLB,No,9.0,3.0,Yes
1,140701,Sanya,Female,58.0,Kolkata,Working Professional,Educational Consultant,NaN,2.0,NaN,NaN,4.0,Less than 5 hours,Moderate,B.Ed,No,6.0,4.0,No
2,140702,Yash,Male,53.0,Jaipur,Working Professional,Teacher,NaN,4.0,NaN,NaN,1.0,7-8 hours,Moderate,B.Arch,Yes,12.0,4.0,No
3,140703,Nalini,Female,23.0,Rajkot,Student,NaN,5.0,NaN,6.84,1.0,NaN,More than 8 hours,Moderate,BSc,Yes,10.0,4.0,No
4,140704,Shaurya,Male,47.0,Kalyan,Working Professional,Teacher,NaN,5.0,NaN,NaN,5.0,7-8 hours,Moderate,BCA,Yes,3.0,4.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93795,234495,Zoya,Female,49.0,Jaipur,Working Professional,Pilot,NaN,3.0,NaN,NaN,5.0,Less than 5 hours,Moderate,BSc,Yes,2.0,2.0,Yes
93796,234496,Shlok,Male,29.0,Ahmedabad,Working Professional,Pilot,NaN,5.0,NaN,NaN,1.0,7-8 hours,Moderate,BE,Yes,11.0,3.0,Yes
93797,234497,Rishi,Male,24.0,Visakhapatnam,Student,NaN,1.0,NaN,7.51,4.0,NaN,7-8 hours,Moderate,B.Tech,No,7.0,1.0,No
93798,234498,Eshita,Female,23.0,Kalyan,Working Professional,Marketing Manager,NaN,4.0,NaN,NaN,2.0,5-6 hours,Healthy,BA,Yes,7.0,5.0,Yes


In [32]:
test_pred  = cb_model.predict(test_data)


In [35]:
submission = pd.DataFrame({
    'id': test['id'],  
    'depression': test_pred
})
submission.to_csv('submission.csv', index=False)
submission

,id,depression
0,140700,0
1,140701,0
2,140702,0
3,140703,1
4,140704,0
...,...,...
93795,234495,0
93796,234496,1
93797,234497,0
93798,234498,1
